# ContractGuard AI v2.0 — Real-World Risk Prediction Training
## Environment: Google Colab T4 GPU

This notebook trains the XGBoost Risk Predictor using the real-world India Road Construction Survey dataset exfiltrated in Phase 4. It accounts for the majority-class violation distribution (77.9%) and utilizes GPU acceleration.

In [ ]:
# Cell 1: Setup
!pip install xgboost imbalanced-learn shap wandb

import pandas as pd
import numpy as np
import xgboost as xgb
import shap
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve
import matplotlib.pyplot as plt
import json

# Upload and unzip payload
from google.colab import files
print("Please upload v2_training_pack.zip if not already present in the sidebar.")
# !unzip v2_training_pack.zip

In [ ]:
# Cell 2: Data Loading
try:
    df = pd.read_csv('execution_merged_real.csv')
except FileNotFoundError:
    # If unzip wasn't run via command
    import zipfile
    with zipfile.ZipFile('v2_training_pack.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    df = pd.read_csv('execution_merged_real.csv')

print(f"Dataset Loaded. Shape: {df.shape}")
print(f"Violation Distribution:\n{df['violation'].value_counts(normalize=True)}")

In [ ]:
# Cell 3: Imbalance Recalibration
# In this dataset, Violation (1) is 77.9%, Compliant (0) is 22.1%.
# We must upsample the COMPLIANT class or adjust scale_pos_weight.

# Feature Selection
X = pd.get_dummies(df[['task_type', 'planned_duration_days', 'actual_duration_days', 'contractor_past_delay_rate', 'is_monsoon_period']])
y = df['violation']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Applying SMOTE to upsample the minority (COMPLIANT) class...")
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print(f"Resampled Training Shape: {X_resampled.shape}")
print(f"Resampled Class Distribution:\n{y_resampled.value_counts(normalize=True)}")

# ALTERNATIVE: Calculate scale_pos_weight for minority-aware training
# weight = count(negative) / count(positive) 
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Calculated scale_pos_weight: {scale_weight:.4f}")

In [ ]:
# Cell 4: GPU Training
print("Initializing XGBoost with T4 GPU support...")

model = xgb.XGBClassifier(
    tree_method='hist', 
    device='cuda',        # Force CUDA T4 usage
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=scale_weight, # Using the calculated weight for majority-flip handling
    random_state=42
)

model.fit(X_resampled, y_resampled)
print("Training Complete.")

In [ ]:
# Cell 5: Evaluation & SHAP
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("--- Classification Report ---")
print(classification_report(y_test, y_pred))
print(f"AUROC: {roc_auc_score(y_test, y_prob):.4f}")

# SHAP Explainability
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print("Generating SHAP Summary Plot...")
shap.summary_plot(shap_values, X_test, plot_type="bar")

In [ ]:
# Cell 6: Export & Download
model.save_model('risk_predictor_v2.json')
print("Model saved to risk_predictor_v2.json")

from google.colab import files
files.download('risk_predictor_v2.json')
print("Artifact download triggered.")